# 第4章: 良い訓練データセットを作るための前処理

この Notebook は、原本 `machine-learning-book/ch04/ch04.ipynb` を最新の pandas / scikit-learn 環境で継続検証できる形に移行したものです。
欠損値処理、カテゴリ変数のエンコード、データ分割、スケーリング、特徴選択まで、章の主要な流れを CI で再実行可能な形に整理しています。


## この Notebook で確認すること

- 現在の `uv` 環境で Python と主要パッケージのバージョンを確認する。
- 原本サブモジュールの図版を読み取り専用で参照する。
- 欠損値処理、カテゴリ変数の変換、one-hot encoding の代表例を再現する。
- Wine データセットで train/test 分割、スケーリング、L1 正則化、逐次特徴選択、ランダムフォレストによる重要度評価を確認する。
- `pytest --nbmake` によるヘッドレス実行で完走することを確認する。


In [ ]:
from importlib.metadata import version
from io import StringIO
from itertools import combinations
from pathlib import Path
import platform
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.datasets import load_wine
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, OneHotEncoder, StandardScaler

PACKAGE_NAMES = ['numpy', 'pandas', 'matplotlib', 'scikit-learn']

def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / '.github/workflows/ci.yml').exists() and (candidate / 'machine-learning-book').exists():
            return candidate
    raise FileNotFoundError('リポジトリルートを特定できませんでした')

REPO_ROOT = find_repo_root(Path.cwd())
FIG_DIR = REPO_ROOT / 'machine-learning-book' / 'ch04' / 'figures'
assert FIG_DIR.exists(), f'図版ディレクトリが見つかりません: {FIG_DIR}'

plt.style.use('seaborn-v0_8-whitegrid')
np.set_printoptions(precision=4, suppress=True)

print(f'Python 実行ファイル: {sys.executable}')
print(f'Python バージョン: {platform.python_version()}')
print(f'Matplotlib バックエンド: {matplotlib.get_backend()}')
print(f'原本 ch04 図版ディレクトリ: {FIG_DIR}')


In [ ]:
package_versions = pd.DataFrame(
    [(name, version(name)) for name in PACKAGE_NAMES],
    columns=['パッケージ', 'バージョン'],
)
package_versions


## 原本図版の参照

書籍の説明で使われる図版アセットは `machine-learning-book/ch04/figures/` に残し、移行版 Notebook からは読み取り専用で利用します。
前処理のコード自体はルート側に移しつつ、章の概念図は原本のまま参照できるようにしています。


In [ ]:
selected_figures = [
    ('04_02.png', 420),
    ('04_03.png', 320),
    ('04_05.png', 520),
    ('04_08.png', 640),
    ('04_09.png', 640),
]

for name, width in selected_figures:
    figure_path = FIG_DIR / name
    print(name)
    display(Image(filename=str(figure_path), width=width))


## 欠損値の扱い

原本と同じ小さな CSV 文字列から DataFrame を作り、欠損値の検出、削除、平均値による補完を確認します。
pandas と scikit-learn の両方で同じ発想を扱えるようにしておきます。


In [ ]:
csv_data = '''A,B,C,D
1.0,2.0,3.0,4.0
5.0,6.0,,8.0
10.0,11.0,12.0,
'''

df_missing = pd.read_csv(StringIO(csv_data))
print('欠損数:')
print(df_missing.isnull().sum())
df_missing


In [ ]:
dropped_rows = df_missing.dropna(axis=0)
dropped_cols = df_missing.dropna(axis=1)

imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
imputed = pd.DataFrame(imputer.fit_transform(df_missing), columns=df_missing.columns)
filled = df_missing.fillna(df_missing.mean(numeric_only=True))

print('行削除後の shape:', dropped_rows.shape)
print('列削除後の shape:', dropped_cols.shape)
print('SimpleImputer と fillna の結果が一致:', np.allclose(imputed.to_numpy(), filled.to_numpy()))
imputed


## カテゴリ変数の処理

次に、順序尺度と名義尺度を含む小さな表で、手動マッピング、`LabelEncoder`、`OneHotEncoder`、`pandas.get_dummies` を比較します。


In [ ]:
df_cat = pd.DataFrame(
    [
        ['green', 'M', 10.1, 'class2'],
        ['red', 'L', 13.5, 'class1'],
        ['blue', 'XL', 15.3, 'class2'],
    ],
    columns=['color', 'size', 'price', 'classlabel'],
)

size_mapping = {'M': 1, 'L': 2, 'XL': 3}
df_cat['size_mapped'] = df_cat['size'].map(size_mapping)

class_le = LabelEncoder()
y = class_le.fit_transform(df_cat['classlabel'].to_numpy())

X_color_label = df_cat[['color']].copy()
color_le = LabelEncoder()
X_color_label['color'] = color_le.fit_transform(X_color_label['color'])

X_for_ohe = df_cat[['color', 'size_mapped', 'price']].copy()

c_transf = ColumnTransformer(
    [('onehot', OneHotEncoder(), [0]), ('pass_size_price', 'passthrough', [1, 2])]
)
encoded = c_transf.fit_transform(X_for_ohe).astype(float)

pd.DataFrame(
    {
        'classlabel_original': df_cat['classlabel'],
        'classlabel_encoded': y,
        'size_mapped': df_cat['size_mapped'],
        'color_label_encoded': X_color_label['color'],
    }
)


In [ ]:
display(pd.get_dummies(df_cat[['price', 'color', 'size']]))
display(pd.get_dummies(df_cat[['price', 'color', 'size']], drop_first=True))

ohe_drop_first = ColumnTransformer(
    [('onehot', OneHotEncoder(drop='first'), [0]), ('pass_size_price', 'passthrough', [1, 2])]
)

print('LabelEncoder 済み color:', X_color_label['color'].to_list())
print('OneHotEncoder の出力 shape:', encoded.shape)
print('drop_first 相当の出力 shape:', ohe_drop_first.fit_transform(X_for_ohe).shape)

df_threshold = df_cat[['color', 'size', 'price', 'classlabel']].copy()
df_threshold['x > M'] = df_threshold['size'].apply(lambda x: 1 if x in {'L', 'XL'} else 0)
df_threshold['x > L'] = df_threshold['size'].apply(lambda x: 1 if x == 'XL' else 0)
df_threshold.drop(columns=['size'])


## Wine データセットの分割とスケーリング

原本では UCI の URL を読んでいましたが、移行版では `scikit-learn` 同梱の Wine データセットを使ってローカル完結にします。
train/test 分割のあと、Min-Max 正規化と標準化を比較します。


In [ ]:
wine = load_wine(as_frame=True)
df_wine = wine.frame.copy()
df_wine.rename(columns={'target': 'Class label'}, inplace=True)
df_wine['Class label'] = df_wine['Class label'] + 1

X = df_wine.drop(columns=['Class label']).to_numpy()
y = df_wine['Class label'].to_numpy()
feature_names = df_wine.drop(columns=['Class label']).columns.to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=0,
    stratify=y,
)

mms = MinMaxScaler()
X_train_norm = mms.fit_transform(X_train)
X_test_norm = mms.transform(X_test)

stdsc = StandardScaler()
X_train_std = stdsc.fit_transform(X_train)
X_test_std = stdsc.transform(X_test)

print('Class labels:', np.unique(y))
print('train/test shape:', X_train.shape, X_test.shape)
df_wine.head()


In [ ]:
ex = np.array([0, 1, 2, 3, 4, 5], dtype=float)
standardized = (ex - ex.mean()) / ex.std()
normalized = (ex - ex.min()) / (ex.max() - ex.min())

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(ex, standardized, marker='o', label='standardized')
ax.plot(ex, normalized, marker='s', label='normalized')
ax.set_xlabel('index')
ax.set_ylabel('scaled value')
ax.set_title('標準化と正規化の比較')
ax.legend(loc='best')
plt.show()
plt.close(fig)

pd.DataFrame({'original': ex, 'standardized': standardized, 'normalized': normalized})


## L1 正則化による疎な解

標準化した Wine データに対して L1 正則化つきロジスティック回帰を学習し、非ゼロ係数の数と `C` による重み変化を確認します。


In [ ]:
lr = OneVsRestClassifier(
    LogisticRegression(
        penalty='l1',
        C=1.0,
        solver='liblinear',
        random_state=0,
        max_iter=200,
    )
)
lr.fit(X_train_std, y_train)

coef_matrix = np.vstack([est.coef_.ravel() for est in lr.estimators_])
intercepts = np.array([est.intercept_[0] for est in lr.estimators_])

print(f'Training accuracy: {lr.score(X_train_std, y_train):.3f}')
print(f'Test accuracy: {lr.score(X_test_std, y_test):.3f}')
print('Intercepts:', intercepts)
print('非ゼロ係数数:', int(np.count_nonzero(coef_matrix)))
coef_matrix


In [ ]:
weights, params = [], []
for c in np.arange(-4.0, 6.0):
    model = OneVsRestClassifier(
        LogisticRegression(
            penalty='l1',
            C=10.0 ** c,
            solver='liblinear',
            random_state=0,
            max_iter=200,
        )
    )
    model.fit(X_train_std, y_train)
    weights.append(model.estimators_[0].coef_.ravel())
    params.append(10.0 ** c)

weights = np.array(weights)

fig, ax = plt.subplots(figsize=(9, 5))
for column in range(weights.shape[1]):
    ax.plot(params, weights[:, column], label=feature_names[column])
ax.axhline(0, color='black', linestyle='--', linewidth=1.5)
ax.set_xscale('log')
ax.set_xlim(10 ** -4, 10 ** 5)
ax.set_xlabel('C (inverse regularization strength)')
ax.set_ylabel('Weight coefficient for class 1')
ax.set_title('L1 正則化下の係数パス')
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5))
fig.tight_layout()
plt.show()
plt.close(fig)


## 逐次後退選択と特徴重要度

原本の `SBS` 実装を現行 API に合わせて再利用し、k-NN に対する逐次後退選択を走らせます。
その後、ランダムフォレストの特徴重要度と `SelectFromModel` による絞り込みも確認します。


In [ ]:
class SBS:
    def __init__(self, estimator, k_features, scoring=accuracy_score, test_size=0.25, random_state=1):
        self.scoring = scoring
        self.estimator = clone(estimator)
        self.k_features = k_features
        self.test_size = test_size
        self.random_state = random_state

    def fit(self, X, y):
        X_train_local, X_test_local, y_train_local, y_test_local = train_test_split(
            X, y, test_size=self.test_size, random_state=self.random_state, stratify=y
        )

        dim = X_train_local.shape[1]
        self.indices_ = tuple(range(dim))
        self.subsets_ = [self.indices_]
        self.scores_ = [self._calc_score(X_train_local, y_train_local, X_test_local, y_test_local, self.indices_)]

        while dim > self.k_features:
            scores = []
            subsets = []
            for p in combinations(self.indices_, r=dim - 1):
                score = self._calc_score(X_train_local, y_train_local, X_test_local, y_test_local, p)
                scores.append(score)
                subsets.append(p)
            best = int(np.argmax(scores))
            self.indices_ = subsets[best]
            self.subsets_.append(self.indices_)
            self.scores_.append(scores[best])
            dim -= 1

        self.k_score_ = self.scores_[-1]
        return self

    def transform(self, X):
        return X[:, self.indices_]

    def _calc_score(self, X_train_local, y_train_local, X_test_local, y_test_local, indices):
        self.estimator.fit(X_train_local[:, indices], y_train_local)
        y_pred = self.estimator.predict(X_test_local[:, indices])
        return self.scoring(y_test_local, y_pred)


In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
sbs = SBS(knn, k_features=1)
sbs.fit(X_train_std, y_train)

k_feat = [len(k) for k in sbs.subsets_]
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(k_feat, sbs.scores_, marker='o')
ax.set_ylim(0.7, 1.02)
ax.set_xlabel('Number of features')
ax.set_ylabel('Accuracy')
ax.set_title('SBS による特徴数と精度の関係')
plt.show()
plt.close(fig)

k3 = list(sbs.subsets_[10])
selected_feature_names = feature_names[k3]
print('3特徴量の候補:', selected_feature_names)

knn.fit(X_train_std, y_train)
full_train_acc = knn.score(X_train_std, y_train)
full_test_acc = knn.score(X_test_std, y_test)

knn.fit(X_train_std[:, k3], y_train)
reduced_train_acc = knn.score(X_train_std[:, k3], y_train)
reduced_test_acc = knn.score(X_test_std[:, k3], y_test)

forest = RandomForestClassifier(n_estimators=200, random_state=1, n_jobs=1)
forest.fit(X_train, y_train)
importances = forest.feature_importances_
indices = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(range(X_train.shape[1]), importances[indices], align='center')
ax.set_xticks(range(X_train.shape[1]))
ax.set_xticklabels(feature_names[indices], rotation=90)
ax.set_xlim(-1, X_train.shape[1])
ax.set_ylabel('Feature importance')
ax.set_title('Random Forest による特徴重要度')
fig.tight_layout()
plt.show()
plt.close(fig)

sfm = SelectFromModel(forest, threshold=0.1, prefit=True)
X_selected = sfm.transform(X_train)

top_features = pd.DataFrame(
    {
        'feature': feature_names[indices],
        'importance': importances[indices],
    }
).head(5)

comparison = pd.DataFrame(
    {
        'setting': ['all features', 'selected 3 features'],
        'train_accuracy': [full_train_acc, reduced_train_acc],
        'test_accuracy': [full_test_acc, reduced_test_acc],
    }
)

display(comparison)
print('threshold=0.1 を満たす特徴数:', X_selected.shape[1])
top_features


## まとめ

第4章の移行版では、原本の前処理フローを以下の形で継続検証可能にしました。

- 欠損値処理、カテゴリ変数の変換、one-hot encoding を小さな表で再現した。
- Wine データセットは `scikit-learn` 同梱データを使い、外部 URL やローカル CSV に依存しないようにした。
- L1 正則化、逐次後退選択、ランダムフォレストによる特徴重要度評価を現在の API で実行可能にした。
- 原本の図版は読み取り専用サブモジュールから再利用し、Notebook 本体は `src/` 側に配置した。
